In [3]:

variables = ['AK', 'BA', 'GB', 'ICT', 'KP', 'PB', 'SD']
colors = ['Red', 'Green', 'Blue']

neighbors = {
    'BA': ['SD', 'PB', 'KP'],
    'SD': ['BA', 'PB'],
    'PB': ['BA', 'SD', 'KP', 'ICT', 'AK'],
    'KP': ['BA', 'PB', 'ICT', 'GB', 'AK'],
    'GB': ['KP', 'AK'],
    'AK': ['GB', 'KP', 'PB'],
    'ICT': ['KP', 'PB']
}

assignments = 0
backtracks = 0


def is_consistent(variable, color, assignment):

    for neighbor in neighbors[variable]:

        if neighbor in assignment:
            if assignment[neighbor] == color:
                return False

    return True


def backtracking_search(assignment):

    global assignments, backtracks
    if len(assignment) == len(variables):
        return assignment.copy()

  
    variable = next(
        v for v in variables if v not in assignment
    )
    for color in colors:

        if is_consistent(variable, color, assignment):

            assignments += 1
            assignment[variable] = color

            result = backtracking_search(assignment)

            if result is not None:
                return result

            del assignment[variable]
            backtracks += 1

    return None

solution = backtracking_search({})

print("Basic Backtracking Solution:")
print(solution)

print("Total Assignments:", assignments)
print("Total Backtracks:", backtracks)


Basic Backtracking Solution:
{'AK': 'Red', 'BA': 'Red', 'GB': 'Green', 'ICT': 'Red', 'KP': 'Blue', 'PB': 'Green', 'SD': 'Blue'}
Total Assignments: 7
Total Backtracks: 0


In [4]:

# Optimized Backtracking Search
# MRV + Degree Heuristic + Forward Checking

variables = ['AK', 'BA', 'GB', 'ICT', 'KP', 'PB', 'SD']

colors = ['Red', 'Green', 'Blue']

neighbors = {
    'BA': ['SD', 'PB', 'KP'],
    'SD': ['BA', 'PB'],
    'PB': ['BA', 'SD', 'KP', 'ICT', 'AK'],
    'KP': ['BA', 'PB', 'ICT', 'GB', 'AK'],
    'GB': ['KP', 'AK'],
    'AK': ['GB', 'KP', 'PB'],
    'ICT': ['KP', 'PB']
}

assignments = 0
backtracks = 0
forward_checks = 0


# Check legal colors
def legal_values(variable, assignment, domains):

    return [
        color for color in domains[variable]
        if all(
            assignment.get(n) != color
            for n in neighbors[variable]
        )
    ]


# MRV + Degree Heuristic
def select_variable(assignment, domains):

    unassigned = [
        v for v in variables if v not in assignment
    ]

    # Find minimum remaining values
    min_values = min(
        len(legal_values(v, assignment, domains))
        for v in unassigned
    )

    candidates = [
        v for v in unassigned
        if len(legal_values(v, assignment, domains))
        == min_values
    ]

    # Degree heuristic for ties
    return max(
        candidates,
        key=lambda v: sum(
            1 for n in neighbors[v]
            if n not in assignment
        )
    )


# Forward Checking
def forward_check(variable, color, assignment, domains):

    global forward_checks

    new_domains = {
        v: list(values)
        for v, values in domains.items()
    }

    for neighbor in neighbors[variable]:

        if neighbor not in assignment:

            if color in new_domains[neighbor]:

                new_domains[neighbor].remove(color)
                forward_checks += 1

                # Check for empty domain
                if len(new_domains[neighbor]) == 0:
                    return None

    return new_domains


# Optimized Backtracking
def optimized_backtracking(assignment, domains):

    global assignments, backtracks

    # Goal test
    if len(assignment) == len(variables):
        return assignment.copy()

    # Select variable using MRV and Degree
    variable = select_variable(assignment, domains)

    # Try legal colors
    for color in legal_values(
        variable, assignment, domains
    ):

        assignments += 1
        assignment[variable] = color

        # Apply Forward Checking
        new_domains = forward_check(
            variable, color, assignment, domains
        )

        if new_domains is not None:

            result = optimized_backtracking(
                assignment, new_domains
            )

            if result is not None:
                return result

        # Backtrack
        del assignment[variable]
        backtracks += 1

    return None


# Initial domains
domains = {
    v: colors.copy() for v in variables
}

# Execute optimized algorithm
solution = optimized_backtracking({}, domains)

print("Optimized Backtracking Solution:")
print(solution)

print("Total Assignments:", assignments)
print("Total Backtracks:", backtracks)
print("Total Forward Checks:", forward_checks)


Optimized Backtracking Solution:
{'KP': 'Red', 'PB': 'Green', 'AK': 'Blue', 'BA': 'Blue', 'GB': 'Green', 'ICT': 'Blue', 'SD': 'Red'}
Total Assignments: 7
Total Backtracks: 0
Total Forward Checks: 11


In [2]:

# =====================================================
# AI-2002: Artificial Intelligence
# Assignment 2 - Part B
# Isolation Game using Alpha-Beta Pruning
# Board Size: 5 x 5
# Search Depth: 4 Plies
# =====================================================

import math

# -------------------------------
# GAME CONFIGURATION
# -------------------------------

SIZE = 5
MAX_DEPTH = 4
WIN_SCORE = 10000

# Queen movement directions
DIRECTIONS = [
    (-1, 0),   # Up
    (1, 0),    # Down
    (0, -1),   # Left
    (0, 1),    # Right
    (-1, -1),  # Upper-left
    (-1, 1),   # Upper-right
    (1, -1),   # Lower-left
    (1, 1)     # Lower-right
]

# Performance counters
nodes_explored = 0
pruned_branches = 0


# =====================================================
# 1. INITIAL GAME STATE
# =====================================================

def initial_state():

    board = [[0 for _ in range(SIZE)]
             for _ in range(SIZE)]

    ai = (0, 0)
    human = (4, 4)

    board[0][0] = 'A'
    board[4][4] = 'H'

    destroyed = set()

    return board, ai, human, destroyed


# =====================================================
# 2. DISPLAY GAME BOARD
# =====================================================

def display_board(ai, human, destroyed):

    print("\nCurrent Board:")

    for r in range(SIZE):

        row = []

        for c in range(SIZE):

            position = (r, c)

            if position == ai:
                row.append(" A ")

            elif position == human:
                row.append(" H ")

            elif position in destroyed:
                row.append(" X ")

            else:
                row.append(" . ")

        print("".join(row))

    print("\nA = AI")
    print("H = Human")
    print("X = Destroyed")
    print(". = Available")


# =====================================================
# 3. GENERATE LEGAL QUEEN MOVES
# =====================================================

def legal_moves(position, opponent, destroyed):

    moves = []

    r, c = position

    for dr, dc in DIRECTIONS:

        nr = r + dr
        nc = c + dc

        while 0 <= nr < SIZE and 0 <= nc < SIZE:

            current = (nr, nc)

            # Cannot cross occupied or destroyed cells
            if current in destroyed or current == opponent:
                break

            moves.append(current)

            nr += dr
            nc += dc

    return moves


# =====================================================
# 4. HEURISTIC FUNCTION 1
# LINEAR MOBILITY DIFFERENCE
# =====================================================

def heuristic1(ai, human, destroyed):

    ai_moves = legal_moves(
        ai, human, destroyed
    )

    human_moves = legal_moves(
        human, ai, destroyed
    )

    return len(ai_moves) - len(human_moves)


# =====================================================
# 5. HEURISTIC FUNCTION 2
# WEIGHTED CELL EVALUATION
# =====================================================

def cell_weight(position):

    r, c = position

    # Central cells receive higher weights
    return 1 + (
        4 - abs(r - 2) - abs(c - 2)
    ) / 4


def heuristic2(ai, human, destroyed):

    ai_moves = legal_moves(
        ai, human, destroyed
    )

    human_moves = legal_moves(
        human, ai, destroyed
    )

    ai_weight = sum(
        cell_weight(move) for move in ai_moves
    )

    human_weight = sum(
        cell_weight(move) for move in human_moves
    )

    return ai_weight - human_weight


# =====================================================
# 6. EVALUATION FUNCTION
# =====================================================

def evaluate(ai, human, destroyed, choice):

    if choice == 1:
        return heuristic1(ai, human, destroyed)

    else:
        return heuristic2(ai, human, destroyed)


# =====================================================
# 7. ALPHA-BETA PRUNING
# MINIMAX SEARCH WITH DEPTH LIMIT
# =====================================================

def alpha_beta(
    ai, human, destroyed,
    depth, alpha, beta,
    maximizing, choice
):

    global nodes_explored, pruned_branches

    nodes_explored += 1

    # Generate moves for current player
    if maximizing:

        moves = legal_moves(
            ai, human, destroyed
        )

        # AI loses if no legal moves
        if not moves:
            return -WIN_SCORE + (MAX_DEPTH - depth), None

    else:

        moves = legal_moves(
            human, ai, destroyed
        )

        # Human loses if no legal moves
        if not moves:
            return WIN_SCORE - (MAX_DEPTH - depth), None

    # Depth cutoff
    if depth == 0:

        score = evaluate(
            ai, human, destroyed, choice
        )

        return score, None

    best_move = None

    # ---------------------------
    # MAXIMIZING PLAYER (AI)
    # ---------------------------

    if maximizing:

        best_score = -math.inf

        for index, move in enumerate(moves):

            # Destroy AI's previous position
            new_destroyed = destroyed | {ai}

            score, _ = alpha_beta(
                move,
                human,
                new_destroyed,
                depth - 1,
                alpha,
                beta,
                False,
                choice
            )

            if score > best_score:

                best_score = score
                best_move = move

            alpha = max(alpha, best_score)

            # Alpha-Beta cutoff
            if alpha >= beta:

                pruned_branches += (
                    len(moves) - index - 1
                )

                break

        return best_score, best_move

    # ---------------------------
    # MINIMIZING PLAYER (HUMAN)
    # ---------------------------

    else:

        best_score = math.inf

        for index, move in enumerate(moves):

            # Destroy Human's previous position
            new_destroyed = destroyed | {human}

            score, _ = alpha_beta(
                ai,
                move,
                new_destroyed,
                depth - 1,
                alpha,
                beta,
                True,
                choice
            )

            if score < best_score:

                best_score = score
                best_move = move

            beta = min(beta, best_score)

            # Alpha-Beta cutoff
            if alpha >= beta:

                pruned_branches += (
                    len(moves) - index - 1
                )

                break

        return best_score, best_move


# =====================================================
# 8. AI TURN
# =====================================================

def ai_turn(ai, human, destroyed, choice):

    global nodes_explored, pruned_branches

    # Reset counters for this turn
    nodes_explored = 0
    pruned_branches = 0

    score, move = alpha_beta(
        ai,
        human,
        destroyed,
        MAX_DEPTH,
        -math.inf,
        math.inf,
        True,
        choice
    )

    print("\n========== AI ANALYSIS ==========")

    print("Evaluation Score:", score)
    print("Nodes Explored:", nodes_explored)
    print("Branches Pruned:", pruned_branches)

    return move


# =====================================================
# 9. HUMAN TURN
# NUMBERED LEGAL MOVES
# =====================================================

def human_turn(ai, human, destroyed):

    moves = legal_moves(
        human, ai, destroyed
    )

    print("\n========== HUMAN TURN ==========")

    if not moves:

        print("No legal moves available!")
        return None

    print("Available moves:")

    for i, move in enumerate(moves, start=1):

        print(f"{i}. {move}")

    while True:

        try:

            choice = int(
                input("\nSelect your move number: ")
            )

            if 1 <= choice <= len(moves):

                return moves[choice - 1]

            else:

                print(
                    "Invalid choice! "
                    "Select a number from the list."
                )

        except ValueError:

            print("Please enter a valid number.")


# =====================================================
# 10. MAIN GAME CONTROLLER
# =====================================================

def play_game():

    board, ai, human, destroyed = initial_state()

    print("\n================================")
    print("         ISOLATION GAME")
    print("================================")

    print("\nSelect Evaluation Heuristic:")

    print("1. Heuristic 1 - Linear Mobility")
    print("2. Heuristic 2 - Weighted Cells")

    while True:

        try:

            choice = int(
                input("\nEnter choice (1 or 2): ")
            )

            if choice in [1, 2]:
                break

            print("Please select 1 or 2.")

        except ValueError:

            print("Enter a valid number.")

    display_board(ai, human, destroyed)

    # Main game loop
    while True:

        # -----------------------
        # AI TURN
        # -----------------------

        ai_moves = legal_moves(
            ai, human, destroyed
        )

        if not ai_moves:

            print("\nAI has no legal moves.")
            print("HUMAN WINS!")

            break

        print("\nAI is thinking...")

        move = ai_turn(
            ai, human, destroyed, choice
        )

        if move is None:

            print("AI has no legal moves.")
            print("HUMAN WINS!")

            break

        # Destroy old AI position
        destroyed.add(ai)

        # Update AI position
        ai = move

        print("\nAI moved to:", ai)

        display_board(ai, human, destroyed)

        # -----------------------
        # HUMAN TURN
        # -----------------------

        human_moves = legal_moves(
            human, ai, destroyed
        )

        if not human_moves:


            print("\nHuman has no legal moves.")
            print("AI WINS!")

            break

        move = human_turn(
            ai, human, destroyed
        )

        if move is None:

            print("HUMAN HAS NO LEGAL MOVES.")
            print("AI WINS!")

            break

        # Destroy old Human position
        destroyed.add(human)

        # Update Human position
        human = move

        print("\nHuman moved to:", human)

        display_board(ai, human, destroyed)


# =====================================================
# 11. PERFORMANCE BENCHMARK
# COMPARE BOTH HEURISTICS
# =====================================================

def benchmark():

    global nodes_explored, pruned_branches

    print("\n================================")
    print("      HEURISTIC COMPARISON")
    print("================================")

    ai = (0, 0)
    human = (4, 4)
    destroyed = set()

    for choice in [1, 2]:

        nodes_explored = 0
        pruned_branches = 0

        score, move = alpha_beta(
            ai,
            human,
            destroyed,
            MAX_DEPTH,
            -math.inf,
            math.inf,
            True,
            choice
        )

        print("\nHeuristic:", choice)

        if choice == 1:
            print("Type: Linear Mobility")

        else:
            print("Type: Weighted Cells")

        print("Best AI Move:", move)
        print("Evaluation Score:", score)
        print("Nodes Explored:", nodes_explored)
        print("Branches Pruned:", pruned_branches)


# =====================================================
# 12. PROGRAM MENU
# =====================================================

print("\n1. Play Isolation Game")
print("2. Compare Both Heuristics")

while True:

    try:

        option = int(
            input("\nSelect option (1 or 2): ")
        )

        if option == 1:

            play_game()
            break

        elif option == 2:

            benchmark()
            break

        else:

            print("Please select 1 or 2.")

    except ValueError:

        print("Enter a valid number.")



1. Play Isolation Game
2. Compare Both Heuristics

      HEURISTIC COMPARISON

Heuristic: 1
Type: Linear Mobility
Best AI Move: (2, 2)
Evaluation Score: 1
Nodes Explored: 2888
Branches Pruned: 3532

Heuristic: 2
Type: Weighted Cells
Best AI Move: (3, 3)
Evaluation Score: 2.0
Nodes Explored: 3324
Branches Pruned: 3744
